# YOLO26 Training - VinBigData Chest X-ray (chi anh co benh)
Pipeline: tai dataset tu Kaggle -> loc chi giu anh co benh (loai class 14 - No finding) -> train YOLO26n voi checkpoint luu tren Google Drive, co the resume tu checkpoint cu.

## 1. Setup moi truong

In [ ]:
!pip install -q ultralytics kaggle
from ultralytics import YOLO
import os, glob, shutil, random
import yaml


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Mount Google Drive (noi luu checkpoint)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Thực tập/VinBigData Chest X-ray Abnormalities Detection/YOLO26 distill/train vinbigdata"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)


Mounted at /content/drive


## 2. Tai dataset tu Kaggle
Upload file `kaggle.json` (API token, lay tu https://www.kaggle.com/settings) khi duoc hoi.

In [ ]:
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload kaggle.json:")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move(list(uploaded.keys())[0], '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)


Upload kaggle.json:


Saving kaggle.json to kaggle.json


In [ ]:
DATA_DIR = "/content/vbd_data"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d awsaf49/vinbigdata-1024-image-dataset -p {DATA_DIR} --unzip
!kaggle datasets download -d awsaf49/vinbigdata-yolo-labels-dataset -p {DATA_DIR} --unzip


Dataset URL: https://www.kaggle.com/datasets/awsaf49/vinbigdata-1024-image-dataset
License(s): CC0-1.0
100% 8.14G/8.14G [01:04<00:00, 135MB/s]

Dataset URL: https://www.kaggle.com/datasets/awsaf49/vinbigdata-yolo-labels-dataset
License(s): unknown
100% 4.85M/4.85M [00:00<00:00, 83.1MB/s]



## 3. Kiem tra cau truc thu muc thuc te
Chay cell nay truoc, neu path o CONFIG (cell duoi) khong khop thi sua lai cho dung.

In [ ]:
for root, dirs, filenames in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, "").count(os.sep)
    if depth > 3:
        continue
    print(root, "->", dirs, "| files sample:", filenames[:3])


/content/vbd_data -> ['vinbigdata', 'labels'] | files sample: ['train_merge.csv']
/content/vbd_data/vinbigdata -> ['train', 'test'] | files sample: ['test.csv', 'train.csv']
/content/vbd_data/vinbigdata/train -> [] | files sample: ['66fccfbfc9d28fc707abb09ce631a42d.png', '4a73cb47410ca27d2349980a1081cc2c.png', 'c7a03a292d749dd79fcabd033a379a39.png']
/content/vbd_data/vinbigdata/test -> [] | files sample: ['c6e3be1241543bb899417700ee878ccc.png', '090a90ca6e01ed9f613578c95c151a01.png', 'de09c63ee469c67331ed80c4ec0d325c.png']
/content/vbd_data/labels -> [] | files sample: ['22141f53eba2088f460be5d3f323121b.txt', '27df67ed0897fb67fe7f2ea4211ead6b.txt', 'eb6c714df22142229464c6b83e47d7d6.txt']


## 4. CONFIG - sua lai neu can sau khi xem cell 3

In [ ]:
# Sua 2 duong dan nay neu cau truc that khac gia dinh
IMG_DIR = f"{DATA_DIR}/vinbigdata/train"          # thu muc chua anh .png, ten file = {image_id}.png
LABEL_DIR = f"{DATA_DIR}/labels" # thu muc chua label .txt YOLO, ten file = {image_id}.txt

CLASS_NAMES = [
    "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
    "Consolidation", "ILD", "Infiltration", "Lung Opacity", "Nodule/Mass",
    "Other lesion", "Pleural effusion", "Pleural thickening", "Pneumothorax",
    "Pulmonary fibrosis",
]  # 14 class benh (class 14 "No finding" bi loai)

VAL_SPLIT = 0.2
SEED = 42
IMG_SIZE = 512


## 5. Loc anh co benh + chia train/val

In [ ]:
# Chi giu anh co it nhat 1 box thuoc class 0-13 (loai class 14 - No finding)
YOLO_ROOT = "/content/YOLO26"
for split in ["train", "val"]:
    os.makedirs(f"{YOLO_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_ROOT}/labels/{split}", exist_ok=True)

disease_image_ids = []
for lbl_path in glob.glob(f"{LABEL_DIR}/*.txt"):
    with open(lbl_path) as f:
        lines = [l for l in f.read().strip().splitlines() if l.strip()]
    disease_lines = [l for l in lines if int(l.split()[0]) != 14]
    if disease_lines:
        image_id = os.path.splitext(os.path.basename(lbl_path))[0]
        disease_image_ids.append((image_id, disease_lines))

print("So anh co benh:", len(disease_image_ids))

random.seed(SEED)
random.shuffle(disease_image_ids)
n_val = int(len(disease_image_ids) * VAL_SPLIT)
splits = {"val": disease_image_ids[:n_val], "train": disease_image_ids[n_val:]}

for split, items in splits.items():
    for image_id, disease_lines in items:
        src_img = f"{IMG_DIR}/{image_id}.png"

        if not os.path.exists(src_img):
            print(src_img)
            continue

        shutil.copy(src_img, f"{YOLO_ROOT}/images/{split}")
        with open(f"{YOLO_ROOT}/labels/{split}/{image_id}.txt", "w") as f:
            f.write("\n".join(disease_lines))
    print(split, ":", len(items), "anh")


So anh co benh: 4394
val : 878 anh
train : 3516 anh


## 6. Tao data.yaml

In [ ]:
yaml_path = f"{YOLO_ROOT}/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump({
        "path": YOLO_ROOT,
        "train": "images/train",
        "val": "images/val",
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }, f)
print(open(yaml_path).read())


names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis
path: /content/YOLO26
train: images/train
val: images/val



## 7. Train YOLO26n - tu resume tu checkpoint cu neu co

In [ ]:
save_dir = f"{DRIVE_PROJECT_DIR}/runs"
run_name = "yolo26n_vbd"
last_ckpt = f"{save_dir}/{run_name}/weights/last.pt"

if os.path.exists(last_ckpt):
    print("Resume tu checkpoint:", last_ckpt)
    model = YOLO(last_ckpt)
    model.train(resume=True)
else:
    print("Train moi tu yolo26n.pt")
    model = YOLO("yolo26n.pt")
    model.train(
        epochs=100,
        imgsz=IMG_SIZE,
        batch=32,
        device=0,
        project=save_dir,
        name=run_name,
        exist_ok=True,
        lr0=0.0054,
        lrf=0.0495,
        momentum=0.947,
        optimizer = "AdamW",
        cls_pw = 0.5,
        weight_decay=0.00064,
        warmup_epochs=0.98,
        box=5.63,
        cls=0.56,
        dfl=9.04,
        mosaic=0.909,
        mixup=0.012,
        copy_paste=0.075,
        scale=0.562,
        fliplr=0.606,
        degrees=1.11,
        shear=1.46,
        translate=0.071,
        hsv_h=0.014,
        hsv_s=0.645,
        hsv_v=0.566,
        bgr=0.106
    )


Train moi tu yolo26n.pt
Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_vbd, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, over

In [ ]:
save_dir = f"{DRIVE_PROJECT_DIR}/runs"
run_name = "yolo26n_vbd"
last_ckpt = f"{save_dir}/{run_name}/weights/last.pt"

if os.path.exists(last_ckpt):
    print("Resume tu checkpoint:", last_ckpt)
    model = YOLO(last_ckpt)
    model.train(data=yaml_path,
        epochs=100,
        imgsz=IMG_SIZE,
        batch=32,
        device=0,
        project=save_dir,
        name=run_name,
        exist_ok=True,)
else:
    print("Train moi tu yolo26n.pt")
    model = YOLO("yolo26n.pt")
    model.train(
        data=yaml_path,
        epochs=100,
        imgsz=IMG_SIZE,
        batch=32,
        device=0,
        project=save_dir,
        name=run_name,
        exist_ok=True,
    )


Resume tu checkpoint: /content/drive/MyDrive/Thực tập/VinBigData Chest X-ray Abnormalities Detection/YOLO26 distill/train vinbigdata/runs/yolo26n_vbd/weights/last.pt
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

KeyboardInterrupt: 

In [ ]:
save_dir = f"{DRIVE_PROJECT_DIR}/runs"
run_name = "yolo26n_vbd"
last_ckpt = f"{save_dir}/{run_name}/weights/last.pt"

if os.path.exists(last_ckpt):
    print("Resume tu checkpoint:", last_ckpt)
    model = YOLO(last_ckpt)
    model.train(data=yaml_path,
        epochs=150,
        imgsz=IMG_SIZE,
        batch=32,
        device=0,
        project=save_dir,
        name=run_name,
        exist_ok=True,
        lr0=0.0054,
        lrf=0.0495,
        momentum=0.947,
        weight_decay=0.00064,
        warmup_epochs=0.98,
        cls_pw = 0.5,
        box=5.63,
        cls=0.56,
        dfl=9.04,
        mosaic=0.909,
        mixup=0.012,
        copy_paste=0.075,
        scale=0.562,
        fliplr=0.606,
        degrees=1.11,
        shear=1.46,
        translate=0.071,
        hsv_h=0.014,
        hsv_s=0.645,
        hsv_v=0.566,
        bgr=0.106)
else:
    print("Train moi tu yolo26n.pt")
    model = YOLO("yolo26n.pt")
    model.train(
        data=yaml_path,
        epochs=100,
        imgsz=IMG_SIZE,
        batch=32,
        device=0,
        project=save_dir,
        name=run_name,
        exist_ok=True,
    )


Resume tu checkpoint: /content/drive/MyDrive/Thực tập/VinBigData Chest X-ray Abnormalities Detection/YOLO26 distill/train vinbigdata/runs/yolo26n_vbd/weights/last.pt
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.106, box=5.63, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.56, cls_pw=0.0, compile=False, conf=None, copy_paste=0.075, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO26/data.yaml, degrees=1.11, deterministic=True, device=0, dfl=9.04, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.606, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.014, hsv_s=0.645, hsv_v=0.566, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0054, lrf=0.0495, mask_ratio=4, max_det=300, mixup=0.012

## 8. Danh gia tren tap val

In [ ]:
best_ckpt = f"{save_dir}/{run_name}/weights/best.pt"
model = YOLO(best_ckpt)
metrics = model.val(data=yaml_path, device=0, verbose=True, visualize=True)

res = metrics.results_dict
print("===== VAL METRICS =====")
print(f"Precision : {res['metrics/precision(B)']:.3f}")
print(f"Recall    : {res['metrics/recall(B)']:.3f}")
print(f"mAP50     : {res['metrics/mAP50(B)']:.3f}")
print(f"mAP50-95  : {res['metrics/mAP50-95(B)']:.3f}")


Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,377,566 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2831.1±507.1 MB/s, size: 451.8 KB)
val: Scanning /content/YOLO26/labels/val.cache... 878 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 878/878 175.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 55/55 2.6it/s 20.9s
                   all        878       7363      0.309        0.3       0.25      0.139
    Aortic enlargement        599       1408      0.388      0.761       0.61      0.412
           Atelectasis         38         51      0.329      0.118      0.175      0.113
         Calcification         94        209      0.157      0.067     0.0657     0.0263
          Cardiomegaly        445       1071      0.367      0.771      0.611      0.456
         Consolidation         70        103  

In [ ]:
best_ckpt = f"{save_dir}/{run_name}/weights/best.pt"
model = YOLO(best_ckpt)
metrics = model.val(data=yaml_path, device=0, verbose=True, visualize=True)

res = metrics.results_dict
print("===== VAL METRICS =====")
print(f"Precision : {res['metrics/precision(B)']:.3f}")
print(f"Recall    : {res['metrics/recall(B)']:.3f}")
print(f"mAP50     : {res['metrics/mAP50(B)']:.3f}")
print(f"mAP50-95  : {res['metrics/mAP50-95(B)']:.3f}")


Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,377,566 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3043.9±975.4 MB/s, size: 482.9 KB)
val: Scanning /content/YOLO26/labels/val.cache... 878 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 878/878 204.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 55/55 3.7s/it 3:25
                   all        878       7381      0.339       0.32       0.28      0.155
    Aortic enlargement        612       1458      0.406      0.719      0.632       0.42
           Atelectasis         39         58      0.366      0.224      0.233      0.113
         Calcification        102        227      0.259      0.163      0.141     0.0681
          Cardiomegaly        488       1167      0.393       0.74      0.624      0.451
         Consolidation         77        110   